In [ ]:
def get_single_erp_params(erp, approx_latency, time_range=0.05, extrema=-1):
    '''
    Return the time and amplitude of the nearest peak in the erp
    at the specified approximate latency, and within the given time range.
    extrema determines whether to find positive or negative peaks.
    '''
    _times = erp.index
    _peaks, _mags = mne.preprocessing.peak_finder(erp, extrema=extrema, verbose='ERROR')
    peak_times = pd.Series(_times[_peaks], index=_peaks)
    _mags = pd.Series(_mags, index=_peaks)
    # find the peaks near to the approximate latency
    nearest_peaks = peak_times[
        (peak_times<approx_latency+time_range)&(peak_times>approx_latency-time_range)
    ]
    # return the time of the largest peak in the vicinity
    return _times[(_mags[nearest_peaks.index]*extrema).idxmax()], (_mags[nearest_peaks.index]*extrema).max()

def get_bootstrapped_erp_params(epochs, sample_size, n_iterations, approx_latency, time_range=0.05, extrema=-1):
    '''
    Return the bootstrapped time and amplitude of the nearest peak in the erp
    at the specified approximate latency, and within the given time range.
    extrema determines whether to find positive or negative peaks.
    '''
    boot_times = []
    boot_mags = []
    boot_erps = []
    for i in range(n_iterations):
        sample_epochs = epochs[np.random.choice(epochs.selection, size=sample_size, replace=True)]
        erp = sample_epochs.average().data[0]
        t, m = get_single_erp_params(pd.Series(erp, index=epochs.times), approx_latency, time_range, extrema)
        boot_times.append(t)
        boot_mags.append(m)
        boot_erps.append(erp)
    return np.array(boot_times), np.array(boot_mags), np.array(boot_erps)



In [11]:
from os import path
from glob import glob
import itertools

from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from scipy.stats import linregress
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import mne

rawdir = '../data/raw/'

triggers = dict(
    S14='BN STD',
    S15='SN DEV',
    S24='SN STD',
    S25='BN DEV',
    S34='CN STD',
    S35='BN DEV',
)

%matplotlib widget

In [2]:
participants = pd.read_csv('participant_details.csv', index_col=0)
delays = participants['Tone_delay'].rename('tone delay')

In [3]:
# list all preprocessed files
# preprocessing involves manual curation, band pass filtering,
# down sampling, removal of eye artifacts, and epoching
files = glob(path.join(rawdir, 'preprocessed', '*'))
print(len(files))
# files

48


# Latency between N100 and MMN

In [8]:
epochs = mne.read_epochs(
    '../data/raw/preprocessed/sub-sd053-epo.fif', verbose='ERROR'
).apply_baseline(verbose='ERROR').to_data_frame()
epochs[['stim', 'kind']] = epochs.condition.map(triggers).str.split(' ', expand=True)
epochs = epochs.drop('condition', axis=1).set_index(['stim', 'kind', 'epoch', 'time']).sort_index()
epochs

Fp1         Fz         F3        F7  \
stim kind epoch time                                                   
BN   DEV  4     -0.300781   1.279897   0.125906   9.585490  2.822604   
                -0.298828  -0.902062  -1.086757   7.873876  0.779283   
                -0.296875  -3.376234  -2.629268   5.486564 -1.306784   
                -0.294922  -5.836605  -4.319858   2.699404 -3.141916   
                -0.292969  -7.990991  -5.971396  -0.172132 -4.478246   
...                              ...        ...        ...       ...   
SN   STD  2319   0.892578  10.415192  28.780272  19.232306 -6.059092   
                 0.894531  10.281085  27.963479  18.938001 -6.572557   
                 0.896484  10.021661  27.184873  18.691623 -7.255580   
                 0.898438   9.732832  26.559816  18.525543 -8.062505   
                 0.900391   9.507792  26.185092  18.449223 -8.950371   

                                 FT9        FC5        FC1         C3  \
stim kind epoch time                                                    
BN   DEV  4     -0.300781   9.000197  42.212234  -2.219007  -1.362891   
                -0.298828   7.903624  37.946681  -3.435025  -2.679941   
                -0.296875   6.597024  33.072720  -5.047430  -4.450418   
                -0.294922   5.161063  28.032330  -6.884083  -6.425192   
                -0.292969   3.680245  23.290205  -8.753438  -8.335462   
...                              ...        ...        ...        ...   
SN   STD  2319   0.892578  10.866399  24.219211  32.024406  41.316674   
                 0.894531  12.903199  23.950616  30.909488  40.701480   
                 0.896484  15.651503  23.464008  29.779203  40.087218   
                 0.898438  18.939682  22.849237  28.781107  39.619135   
                 0.900391  22.522496  22.184165  28.031657  39.400289   

                                  T7       TP9  ...         C6         C2  \
stim kind epoch time                            ...                         
BN   DEV  4     -0.300781  -8.944509 -4.317166  ...   4.009191  -8.723368   
                -0.298828 -10.127294 -3.821534  ...   3.606309  -9.300628   
                -0.296875 -10.579579 -3.226570  ...   2.456177 -10.090395   
                -0.294922 -10.300925 -2.609881  ...   0.726030 -10.992112   
                -0.292969  -9.391406 -2.046531  ...  -1.358131 -11.899243   
...                              ...       ...  ...        ...        ...   
SN   STD  2319   0.892578  51.304757 -2.803168  ...  20.727369  41.751742   
                 0.894531  50.409634 -1.241770  ...  19.880133  40.586436   
                 0.896484  49.451891  0.177926  ...  19.001476  39.579103   
                 0.898438  48.506202  1.252789  ...  18.250267  38.863419   
                 0.900391  47.654220  1.840037  ...  17.758544  38.529197   

                                 FC4        FT8         F6        AF8  \
stim kind epoch time                                                    
BN   DEV  4     -0.300781  -4.586885  -1.804788  13.933810 -21.404715   
                -0.298828  -5.561084  -4.848037  10.166273 -21.286168   
                -0.296875  -7.145926  -7.988140   5.247258 -20.451677   
                -0.294922  -9.139666 -10.870453  -0.389211 -18.992092   
                -0.292969 -11.304217 -13.185574  -6.250679 -17.065519   
...                              ...        ...        ...        ...   
SN   STD  2319   0.892578  24.799686  14.393931   4.431223  -1.622995   
                 0.894531  23.473750  13.475835   3.748932  -3.215549   
                 0.896484  21.583749  12.431231   2.487739  -4.876869   
                 0.898438  19.393616  11.401381   0.787439  -6.422135   
                 0.900391  17.186255  10.510259  -1.161023  -7.672443   

                                 AF4         F2        FCz         Cz  
stim kind epoch time                                                   
BN   DEV  4     -0.300781   3.431162   4.119716  -

In [15]:
erps = {}
for stim, kind in itertools.product(['SN', 'BN'], ['STD', 'DEV']):
    erps[(stim, kind)] = epochs.loc[(stim, kind)].groupby('time').mean()
erps = pd.concat(erps)
erps

Fp1        Fz        F3        F7       FT9       FC5  \
       time                                                                    
SN STD -0.300781  0.546315  1.066837  1.003176  0.919238  0.223984  0.804194   
       -0.298828  0.527454  1.036925  0.990336  0.884269  0.171953  0.755286   
       -0.296875  0.498225  0.982972  0.954742  0.820708  0.107460  0.690785   
       -0.294922  0.460298  0.906990  0.897148  0.735750  0.033940  0.615198   
       -0.292969  0.416079  0.812262  0.819280  0.637878 -0.045720  0.533354   
...                    ...       ...       ...       ...       ...       ...   
BN DEV  0.892578 -0.029219 -1.958411 -2.652140  0.071844  0.675750 -3.838195   
        0.894531  0.089469 -1.788364 -2.577526  0.083153  0.807280 -3.772990   
        0.896484  0.169581 -1.676143 -2.533284  0.028653  0.836043 -3.731635   
        0.898438  0.208682 -1.619391 -2.513877 -0.084947  0.758030 -3.709040   
        0.900391  0.208683 -1.609915 -2.511262 -0.245275  0.581277 -3.697894   

                       FC1        C3        T7       TP9  ...        C6  \
       time                                               ...             
SN STD -0.300781  1.120844  1.145826  0.887289 -0.097634  ...  0.956671   
       -0.298828  1.088196  1.124899  0.972002 -0.169868  ...  0.922348   
       -0.296875  1.036337  1.077197  1.024453 -0.231033  ...  0.861499   
       -0.294922  0.966402  1.000832  1.041759 -0.273770  ...  0.777673   
       -0.292969  0.880285  0.896567  1.023897 -0.292415  ...  0.675968   
...                    ...       ...       ...       ...  ...       ...   
BN DEV  0.892578 -0.501414 -0.632484 -3.616435 -0.399601  ... -0.738811   
        0.894531 -0.363510 -0.494078 -3.518461 -0.432778  ... -0.619187   
        0.896484 -0.278027 -0.387262 -3.463931 -0.442620  ... -0.509638   
        0.898438 -0.241637 -0.312900 -3.459125 -0.432565  ... -0.411517   
        0.900391 -0.246225 -0.267375 -3.504172 -0.407010  ... -0.326305   

                        C2       FC4       FT8        F6       AF8       AF4  \
       time                                                                    
SN STD -0.300781  0.973995  1.147228  0.574269  0.665183  0.558034  0.906826   
       -0.298828  0.928026  1.100707  0.502381  0.631728  0.593296  0.872846   
       -0.296875  0.857689  1.019625  0.423325  0.578175  0.597307  0.812031   
       -0.294922  0.766224  0.909007  0.342377  0.507224  0.568517  0.728523   
       -0.292969  0.658498  0.775911  0.264257  0.422734  0.508989  0.627729   
...                    ...       ...       ...       ...       ...       ...   
BN DEV  0.892578 -0.385749 -1.205071 -2.142879 -1.063560 -1.364520 -1.488557   
        0.894531 -0.203499 -1.045270 -2.062226 -0.925635 -1.287201 -1.372313   
        0.896484 -0.055624 -0.901879 -1.981952 -0.838369 -1.236183 -1.289006   
        0.898438  0.057341 -0.777118 -1.903922 -0.796808 -1.210157 -1.234474   
        0.900391  0.137997 -0.671624 -1.830928 -0.791335 -1.204106 -1.201171   

                        F2       FCz        Cz  
       time                                     
SN STD -0.300781  0.943303  0.994418  1.007769  
       -0.298828  0.905959  0.956907  0.942594  
       -0.296875  0.847308  0.901778  0.850783  
       -0.294922  0.770279  0.830818  0.737375  
       -0.292969  0.678913  0.746520  0.608909  
...                    ...       ...       ...  
BN DEV  0.892578 -2.119682 -0.799851 -0.206213  
        0.894531 -1.949584 -0.612711 -0.015070  
        0.896484 -1.824833 -0.477761  0.129560  
        0.898438 -1.744518 -0.394970  0.224448  
        0.900391 -1.702631 -0.358910  0.271753  

[2464 rows x 64 columns]